# CLIP Text Embeddings

Encode text, compare vectors, and inspect a two-dimensional UMAP projection.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import umap
from transformers import CLIPModel, CLIPProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "openai/clip-vit-base-patch32"
model = CLIPModel.from_pretrained(model_name).to(device).eval()
processor = CLIPProcessor.from_pretrained(model_name)

In [ ]:
texts = ["king", "queen", "man", "woman", "boy", "girl", "prince", "princess"]
inputs = processor(text=texts, return_tensors="pt", padding=True, truncation=True).to(device)
with torch.inference_mode():
    features = model.get_text_features(**inputs)
    features = features / features.norm(dim=-1, keepdim=True)

print("Embedding shape:", tuple(features.shape))
features[0].cpu().tolist()[:10]

In [ ]:
vectors = features.cpu().numpy()
lookup = dict(zip(texts, vectors, strict=True))
result = lookup["queen"] + lookup["king"] - lookup["man"]
result /= np.linalg.norm(result)
sorted(zip(texts, vectors @ result, strict=True), key=lambda row: row[1], reverse=True)

In [ ]:
points = umap.UMAP(n_neighbors=5, min_dist=0.2, random_state=42).fit_transform(np.vstack([vectors, result]))
labels = texts + ["queen + king - man"]
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(points[:, 0], points[:, 1])
for point, label in zip(points, labels, strict=True):
    ax.annotate(label, point, xytext=(4, 4), textcoords="offset points")
ax.set(title="CLIP text arithmetic (UMAP)", xticks=[], yticks=[]);